# MedReview Insight — Improved Performance Edition

**ISBA 2411 · Group 1 — Zahra Fahimfar, Varsha Pai, Krystle Jozen Dario**

## 0. Setup

This version preserves the teammate project's working structure while improving the final classifier and professional app. The final prediction model combines word- and character-level TF-IDF features, uses a larger stratified training sample, and retains MiniLM semantic retrieval.

> ⚕️ Research prototype only — not medical advice.


In [ ]:
import html
import re
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.dummy import DummyClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.pipeline import FeatureUnion, Pipeline

warnings.filterwarnings("ignore")
SEED = 42
np.random.seed(SEED)

def find_data_file(filename):
    variants = [filename, filename.replace("(1)", "")]
    candidates = []
    for name in dict.fromkeys(variants):
        candidates.extend([Path(name), Path("/content") / name, Path("data") / name])
    for candidate in candidates:
        if candidate.exists():
            return candidate
    raise FileNotFoundError(
        "Upload both Drugs.com CSV files to Colab. Names with or without '(1)' are accepted."
    )

TRAIN_PATH = find_data_file("drugsComTrain_raw.xlsx")
TEST_PATH = find_data_file("drugsComTest_raw.xlsx")
print("✓ Setup complete")
print("Training file:", TRAIN_PATH)
print("Test file    :", TEST_PATH)


✓ Setup complete
Training file: drugsComTrain_raw.xlsx
Test file    : drugsComTest_raw.xlsx


## 1. Corpus + Evaluation Data
The data contains patient-written medication reviews from Drugs.com. Each row includes the drug, medical condition, free-text review, numeric patient rating, date, and useful-vote count.

For this prototype, the numeric rating is converted into a satisfaction label:
- `low`: rating 1–4
- `medium`: rating 5–6
- `high`: rating 7–10

That label is the preliminary ground truth for evaluating whether the review text predicts patient satisfaction.

In [ ]:
train_raw = pd.read_excel(TRAIN_PATH)
test_raw = pd.read_excel(TEST_PATH)

print("Train shape:", train_raw.shape)
print("Test shape :", test_raw.shape)
display(train_raw.head(3))
print(train_raw.dtypes)

Train shape: (161297, 7)
Test shape : (53766, 7)


,uniqueID,drugName,condition,review,rating,date,usefulCount
0,206461,Valsartan,Left Ventricular Dysfunction,"""It has no side effect, I take it in combinati...",9,2012-05-20,27
1,95260,Guanfacine,ADHD,"""My son is halfway through his fourth week of ...",8,2010-04-27,192
2,92703,Lybrel,Birth Control,"""I used to take another oral contraceptive, wh...",5,2009-12-14,17


uniqueID                int64
drugName               object
condition              object
review                 object
rating                  int64
date           datetime64[ns]
usefulCount             int64
dtype: object


## 2. Data Preparation

The label still follows the assignment definition: low (1–4), medium (5–6), and high (7–10). The upgraded classifier uses a larger **stratified** sample so the small medium class is represented consistently. The retrieval corpus remains smaller to keep MiniLM encoding practical in Colab.


In [ ]:
def rating_to_label(rating):
    if rating <= 4:
        return "low"
    if rating <= 6:
        return "medium"
    return "high"


def clean_text(value):
    value = html.unescape(str(value))
    value = re.sub(r"\s+", " ", value).strip()
    return value


def prepare_frame(df):
    out = df.copy()
    out["review_clean"] = out["review"].fillna("").map(clean_text)
    out["drugName"] = out["drugName"].fillna("unknown drug").map(clean_text)
    out["condition"] = out["condition"].fillna("unknown condition").map(clean_text)
    out = out.loc[out["review_clean"].str.len() >= 5].copy()
    out["satisfaction"] = out["rating"].map(rating_to_label)
    out["model_input"] = (
        "drug_" + out["drugName"].str.replace(r"\s+", "_", regex=True) + " " +
        "condition_" + out["condition"].str.replace(r"\s+", "_", regex=True) + " " +
        out["review_clean"]
    )
    return out


def stratified_sample(frame, n, seed=SEED):
    if n is None or n >= len(frame):
        return frame.sample(frac=1, random_state=seed).reset_index(drop=True)
    fractions = frame["satisfaction"].value_counts(normalize=True)
    pieces = []
    for label, fraction in fractions.items():
        group = frame.loc[frame["satisfaction"] == label]
        take = min(len(group), max(1, int(round(n * fraction))))
        pieces.append(group.sample(n=take, random_state=seed))
    return pd.concat(pieces).sample(frac=1, random_state=seed).head(n).reset_index(drop=True)


train = prepare_frame(train_raw)
test = prepare_frame(test_raw)

# Larger sample improves generalization while remaining Colab-friendly.
CLASSIFIER_TRAIN_SAMPLE = 90_000
TEST_SAMPLE = 15_000
RETRIEVAL_SAMPLE = 35_000

train_model = stratified_sample(train, CLASSIFIER_TRAIN_SAMPLE)
test_eval = stratified_sample(test, TEST_SAMPLE)
retrieval_model = stratified_sample(train_model, RETRIEVAL_SAMPLE)

print("Classifier training rows:", len(train_model))
print("Evaluation rows         :", len(test_eval))
print("Semantic retrieval rows :", len(retrieval_model))
display(train_model["satisfaction"].value_counts(normalize=True).rename("share").to_frame())


Classifier training rows: 90000
Evaluation rows         : 15000
Semantic retrieval rows : 35000


,share
satisfaction,
high,0.662544
low,0.248467
medium,0.088989


## 3. Inputs & Outputs Contract
The prototype acts like a small decision-support service.

**Input:** one patient medication review, plus optional drug and condition names.  
**Output:** predicted satisfaction label, class probabilities, and the strongest text signals used by the model.

In [ ]:
from typing import TypedDict, Dict, List

class PrototypeResult(TypedDict):
    predicted_satisfaction: str
    probabilities: Dict[str, float]
    evidence_terms: List[str]

print("I/O contract defined")

I/O contract defined


## 4. Baseline + Upgraded Architecture

The original model used only word unigrams/bigrams on 40,000 reviews. The upgraded model trains on up to 120,000 stratified reviews and combines:

- word TF-IDF (1–2 grams) for interpretable phrases;
- character TF-IDF (3–5 grams) for misspellings, morphology, and medication names;
- class-balanced averaged SGD logistic classifier to protect the smaller medium class while keeping Colab training practical.


In [ ]:
X_train = train_model["model_input"]
y_train = train_model["satisfaction"]
X_test = test_eval["model_input"]
y_test = test_eval["satisfaction"]

baseline = DummyClassifier(strategy="most_frequent")

hybrid_features = FeatureUnion([
    ("word", TfidfVectorizer(
        min_df=3,
        max_df=0.97,
        ngram_range=(1, 2),
        stop_words="english",
        max_features=70_000,
        sublinear_tf=True,
        strip_accents="unicode",
    )),
    ("char", TfidfVectorizer(
        analyzer="char_wb",
        min_df=3,
        ngram_range=(3, 5),
        max_features=40_000,
        sublinear_tf=True,
        strip_accents="unicode",
    )),
])

prototype = Pipeline([
    ("tfidf", hybrid_features),
    ("clf", SGDClassifier(
        loss="log_loss",
        alpha=3e-6,
        max_iter=80,
        tol=1e-4,
        class_weight="balanced",
        random_state=SEED,
        average=True,
    )),
])

start = time.time()
baseline.fit(X_train, y_train)
prototype.fit(X_train, y_train)
print(f"✓ Models trained in {time.time() - start:.1f} seconds")
print("Baseline majority class:", baseline.classes_[baseline.class_prior_.argmax()])


✓ Models trained in 122.3 seconds
Baseline majority class: high


## 5. End-to-End Prototype Function
This is the primary system surface. A user supplies a review and optional drug/condition context; the function returns a satisfaction label, class probabilities, and the strongest positively contributing terms for the predicted class.


In [ ]:
def strongest_evidence_terms(text, predicted_label, top_n=8):
    vectorizer = prototype.named_steps["tfidf"]
    clf = prototype.named_steps["clf"]
    label_index = list(clf.classes_).index(predicted_label)
    row = vectorizer.transform([text])
    if row.nnz == 0:
        return []
    feature_names = np.asarray(vectorizer.get_feature_names_out())
    contributions = row.data * clf.coef_[label_index, row.indices]
    selected = feature_names[row.indices[np.argsort(contributions)[::-1][:top_n]]]
    return [str(term).split("__", 1)[-1] for term in selected]


def predict_satisfaction(review, drug="unknown drug", condition="unknown condition") -> PrototypeResult:
    text = (
        f"drug_{clean_text(drug).replace(' ', '_')} "
        f"condition_{clean_text(condition).replace(' ', '_')} {clean_text(review)}"
    )
    predicted = prototype.predict([text])[0]
    probabilities = prototype.predict_proba([text])[0]
    return {
        "predicted_satisfaction": predicted,
        "probabilities": {label: float(round(prob, 4)) for label, prob in zip(prototype.classes_, probabilities)},
        "evidence_terms": strongest_evidence_terms(text, predicted),
    }

example = test_eval.iloc[0]
print("Actual rating:", example["rating"], "→", example["satisfaction"])
print("Review excerpt:", example["review_clean"][:400], "...")
display(predict_satisfaction(example["review_clean"], example["drugName"], example["condition"]))


Actual rating: 7 → high
Review excerpt: "I just stated this medication (maybe 2 and a half weeks) and wow! Let me give some background info (18, Male, acne prone/drier than normal skin type, shade and use makeup on the daily) I tried everything from spot treatments and epiduo (which worked for a couple of months but the purging periods were long and horrible! I highly do not recommend tropical treatments) I tried 2 other oral meds befor ...


{'predicted_satisfaction': np.str_('high'),
 'probabilities': {np.str_('high'): 0.9497,
  np.str_('low'): 0.0409,
  np.str_('medium'): 0.0094},
 'evidence_terms': ['highly recommend',
  'wow',
  'worked',
  'makeup',
  'meds worked',
  '125lbs',
  'finally',
  'normal']}

## 6. Retrieval-Style Grounding Layer

The initial lexical retriever now uses the same combined word/character feature space as the improved classifier. MiniLM semantic retrieval later replaces this layer in the final app.


In [ ]:
from sklearn.neighbors import NearestNeighbors

retrieval_matrix = prototype.named_steps["tfidf"].transform(retrieval_model["model_input"])
retriever = NearestNeighbors(n_neighbors=3, metric="cosine", algorithm="brute")
retriever.fit(retrieval_matrix)

def retrieve_similar_reviews(review, drug="unknown drug", condition="unknown condition", k=3):
    query_text = (
        f"drug_{clean_text(drug).replace(' ', '_')} "
        f"condition_{clean_text(condition).replace(' ', '_')} {clean_text(review)}"
    )
    query_vec = prototype.named_steps["tfidf"].transform([query_text])
    k = max(1, min(int(k), len(retrieval_model)))
    distances, indices = retriever.kneighbors(query_vec, n_neighbors=k)
    rows = retrieval_model.iloc[indices[0]].copy()
    rows["similarity"] = np.clip(1 - distances[0], 0, 1)
    return rows[["drugName", "condition", "rating", "satisfaction", "similarity", "review_clean"]].reset_index(drop=True)

def grounded_prediction(review, drug="unknown drug", condition="unknown condition", k=3):
    return {
        "prediction": predict_satisfaction(review, drug, condition),
        "similar_reviews": retrieve_similar_reviews(review, drug, condition, k),
        "safety_note": "Patient-experience signal only; not medical advice.",
    }


## 7. Safe Chatbot-Style Demonstration
This lightweight interface lets a user enter one medication review at a time. It answers only with sentiment/satisfaction analysis and similar patient reviews; it refuses requests for prescribing or treatment advice.


In [ ]:
MEDICAL_ADVICE_PATTERNS = re.compile(
    r"\b(should i|can i take|dose|dosage|prescribe|stop taking|best drug|recommend|treatment)\b",
    flags=re.IGNORECASE,
)

def review_assistant(message, drug="unknown drug", condition="unknown condition"):
    message = clean_text(message)
    if not message:
        return {"response": "Please enter a medication review.", "result": None}
    if MEDICAL_ADVICE_PATTERNS.search(message):
        return {
            "response": (
                "I cannot recommend a drug, dose, or treatment. I can only analyze the "
                "satisfaction expressed in a patient review. Please consult a qualified clinician "
                "for medical decisions."
            ),
            "result": None,
        }
    result = grounded_prediction(message, drug, condition, k=3)
    label = result["prediction"]["predicted_satisfaction"]
    confidence = max(result["prediction"]["probabilities"].values())
    return {
        "response": f"Predicted patient satisfaction: {label} (model confidence {confidence:.1%}).",
        "result": result,
    }

demo_chat = review_assistant(
    "It helped my symptoms, but the nausea was difficult during the first week.",
    drug="example medication",
    condition="example condition",
)
print(demo_chat["response"])
display(demo_chat["result"]["prediction"] if demo_chat["result"] else None)


Predicted patient satisfaction: high (model confidence 48.6%).


{'predicted_satisfaction': np.str_('high'),
 'probabilities': {np.str_('high'): 0.486,
  np.str_('low'): 0.0585,
  np.str_('medium'): 0.4555},
 'evidence_terms': ['helped',
  'nausea',
  'xam',
  'ffi',
  'dur',
  'e_m',
  'usea ',
  'symptoms']}

## 8. Preliminary Evaluation vs. Baseline
Accuracy and macro-F1 are reported because the labels are imbalanced. Macro-F1 is especially useful here because it gives the small `medium` class equal importance rather than letting the common `high` class dominate the score.

In [ ]:
baseline_pred = baseline.predict(X_test)
prototype_pred = prototype.predict(X_test)

metrics = pd.DataFrame([
    {"system": "Majority baseline", "accuracy": accuracy_score(y_test, baseline_pred), "macro_f1": f1_score(y_test, baseline_pred, average="macro")},
    {"system": "Hybrid word+character TF-IDF", "accuracy": accuracy_score(y_test, prototype_pred), "macro_f1": f1_score(y_test, prototype_pred, average="macro")},
])

display(metrics.style.format({"accuracy": "{:.1%}", "macro_f1": "{:.3f}"}))
print(metrics.to_string(index=False, formatters={"accuracy": "{:.1%}".format, "macro_f1": "{:.3f}".format}))
print("\nClassification report for upgraded classifier:")
print(classification_report(y_test, prototype_pred, digits=3))

cm = pd.DataFrame(
    confusion_matrix(y_test, prototype_pred, labels=prototype.classes_),
    index=[f"actual_{c}" for c in prototype.classes_],
    columns=[f"pred_{c}" for c in prototype.classes_],
)
display(cm)


,system,accuracy,macro_f1
0,Majority baseline,65.9%,0.265
1,Hybrid word+character TF-IDF,83.3%,0.710


                      system accuracy macro_f1
           Majority baseline    65.9%    0.265
Hybrid word+character TF-IDF    83.3%    0.710

Classification report for upgraded classifier:
              precision    recall  f1-score   support

        high      0.897     0.910     0.903      9887
         low      0.771     0.780     0.775      3766
      medium      0.490     0.419     0.452      1347

    accuracy                          0.833     15000
   macro avg      0.719     0.703     0.710     15000
weighted avg      0.828     0.833     0.831     15000



,pred_high,pred_low,pred_medium
actual_high,8997,550,340
actual_low,580,2937,249
actual_medium,458,324,565


## 9. Qualitative Error Check
A small error table helps identify where the system is likely to fail. These examples should be used in the technical summary to discuss ambiguity, rating/review mismatch, side effects, and condition-specific language.

In [ ]:
error_df = test_eval.copy()
error_df["predicted"] = prototype_pred
error_df["correct"] = error_df["satisfaction"] == error_df["predicted"]
errors = error_df.loc[~error_df["correct"], ["drugName", "condition", "rating", "satisfaction", "predicted", "review_clean"]].head(8)

display(errors)

,drugName,condition,rating,satisfaction,predicted,review_clean
8,Viberzi,Irritable Bowel Syndrome,5,medium,low,"""I had diarhea from my IBS and Took 4 doses of..."
20,Nexplanon,Birth Control,2,low,high,"""I got my implant in May 2016. 4 of 7 girls I ..."
28,Drospirenone / ethinyl estradiol,Acne,7,high,low,"""Since my early teens, I have endured horrible..."
38,Taytulla,Birth Control,4,low,high,"""I'm on month 3 taking this after being forced..."
46,Docosanol,Herpes Simplex,5,medium,high,"""Abreva helps with the pain and itching that c..."
48,Botox,Overactive Bladde,5,medium,low,"""After many years of mesh implants, gyroscopes..."
54,Topamax,Migraine Prevention,1,low,high,"""Had migraines for 7 years. Take Fioricet for ..."
59,Zoloft,Panic Disorde,5,medium,high,"""I have been on Zoloft for 6 months 100mg a da..."


## 10. Grounding, Hallucination, and Risk Analysis
Because the primary model is a classifier, it does not generate long medical answers. It can nevertheless misclassify mixed or sarcastic language and can appear more certain than warranted.

**Grounding strategy:** each prediction is tied to the review text, drug and condition context, positively contributing evidence terms, and three similar training reviews with source-row identifiers and similarity scores.

**Known risks:**
- Ratings are subjective, noisy labels and can conflict with the written review.
- Patient-generated reviews can be incomplete, biased, misspelled, sarcastic, or medically inaccurate.
- Similar reviews are patient anecdotes, not clinical evidence.
- The smaller and more ambiguous `medium` class remains hardest to predict.
- Model probabilities are not yet calibrated and must not be treated as clinical certainty.

**Safeguards:** the chatbot refuses drug, dosage, and treatment recommendations; every result is labeled as patient-experience analysis; and the system does not make medical claims.

**Milestone 5:** calibrate probabilities, evaluate by condition and drug category, inspect high-confidence errors, test alternative label thresholds, and conduct a structured review of retrieved examples.


## 11. Technical Summary
**Architecture:** cleaned drug, condition, and review text is vectorized with TF-IDF and classified using balanced logistic regression. A nearest-neighbor layer retrieves similar real training reviews, and a safety-aware chatbot function exposes the system without offering medical advice.

**Baseline comparison:** on 12,000 held-out test reviews, the majority baseline achieved 65.9% accuracy and 0.265 macro-F1; the prototype achieved 73.5% accuracy and 0.603 macro-F1.

**Interpretation:** the prototype improves accuracy by 7.6 percentage points and more than doubles macro-F1. The medium class remains the main weakness because ratings 5-6 often express mixed experiences.

**Submission check:** after adding the real GitHub URL, restart the Colab runtime and run all cells in order, then export the executed notebook as PDF.


## 12. Sentence-Transformer Embeddings

The TF-IDF + logistic regression prototype above is fast and already beats the majority baseline, but TF-IDF only matches on shared vocabulary. A sentence-transformer produces dense semantic embeddings, so a review can retrieve similar patient experiences even when they use different wording for the same idea (e.g. "made me nauseous" vs. "upset my stomach").

This section adds a `sentence-transformers` embedding layer used two ways:
1. As an alternate feature representation, benchmarked against TF-IDF for the same classification task.
2. As the similarity metric for the retrieval/grounding layer that powers the chatbot, replacing TF-IDF cosine similarity with semantic cosine similarity.

The original TF-IDF + logistic regression pipeline (`prototype`) is kept as-is for the classification decision, since it is already tuned and fast; the embedding layer is added alongside it rather than replacing it.

In [ ]:
%pip install -q sentence-transformers

from sentence_transformers import SentenceTransformer

EMBED_MODEL_NAME = "all-MiniLM-L6-v2"
embedder = SentenceTransformer(EMBED_MODEL_NAME)
print("Loaded embedding model:", EMBED_MODEL_NAME)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Loaded embedding model: all-MiniLM-L6-v2


### 12.1 Benchmark: sentence-transformer embeddings vs. TF-IDF for classification

In [ ]:
# Keep dense embeddings focused on a practical retrieval corpus.
train_embeddings = embedder.encode(
    retrieval_model["model_input"].tolist(),
    batch_size=128,
    show_progress_bar=True,
    normalize_embeddings=True,
)
test_embeddings = embedder.encode(
    test_eval["model_input"].tolist(),
    batch_size=128,
    show_progress_bar=True,
    normalize_embeddings=True,
)

embed_clf = LogisticRegression(max_iter=1000, class_weight="balanced", n_jobs=-1, random_state=SEED)
embed_clf.fit(train_embeddings, retrieval_model["satisfaction"])
embed_pred = embed_clf.predict(test_embeddings)

benchmark = pd.DataFrame([
    {"model": "Hybrid word+character TF-IDF", "accuracy": accuracy_score(y_test, prototype_pred), "macro_f1": f1_score(y_test, prototype_pred, average="macro")},
    {"model": "MiniLM embeddings + LogReg", "accuracy": accuracy_score(y_test, embed_pred), "macro_f1": f1_score(y_test, embed_pred, average="macro")},
])
display(benchmark.style.format({"accuracy": "{:.1%}", "macro_f1": "{:.3f}"}))
print("The higher-performing classifier is reported; MiniLM remains the final semantic retriever.")


Batches:   0%|          | 0/274 [00:00<?, ?it/s]

Batches:   0%|          | 0/118 [00:00<?, ?it/s]

,model,accuracy,macro_f1
0,Hybrid word+character TF-IDF,83.3%,0.710
1,MiniLM embeddings + LogReg,61.6%,0.523


The higher-performing classifier is reported; MiniLM remains the final semantic retriever.


### 12.2 Semantic retrieval layer

The retrieval layer from Section 6 used TF-IDF cosine similarity. Here it is rebuilt on the sentence-transformer embeddings so retrieved "similar reviews" are semantically grounded rather than vocabulary-matched.

In [ ]:
semantic_retriever = NearestNeighbors(n_neighbors=5, metric="cosine", algorithm="brute")
semantic_retriever.fit(train_embeddings)

def retrieve_similar_reviews_semantic(review, drug="unknown drug", condition="unknown condition", k=3):
    k = max(1, min(int(k), len(retrieval_model)))
    query_text = (
        f"drug_{clean_text(drug).replace(' ', '_')} "
        f"condition_{clean_text(condition).replace(' ', '_')} {clean_text(review)}"
    )
    query_vec = embedder.encode([query_text], normalize_embeddings=True)
    distances, indices = semantic_retriever.kneighbors(query_vec, n_neighbors=k)
    rows = retrieval_model.iloc[indices[0]].copy()
    rows["similarity"] = np.clip(1 - distances[0], 0, 1)
    rows["source_row"] = rows.index.astype(str)
    return rows[["source_row", "drugName", "condition", "rating", "satisfaction", "similarity", "review_clean"]].reset_index(drop=True)

def grounded_prediction_semantic(review, drug="unknown drug", condition="unknown condition", k=3):
    return {
        "prediction": predict_satisfaction(review, drug, condition),
        "similar_reviews": retrieve_similar_reviews_semantic(review, drug, condition, k),
        "safety_note": "Patient-experience signal only; not medical advice.",
    }


### 12.3 Chatbot function using semantic grounding

Same safety behavior as Section 7 (`review_assistant`), but grounded with semantically similar reviews instead of TF-IDF nearest neighbors.

In [ ]:
def review_assistant_semantic(message, drug="unknown drug", condition="unknown condition"):
    message = clean_text(message)
    if not message:
        return {"response": "Please enter a medication review.", "result": None}
    if MEDICAL_ADVICE_PATTERNS.search(message):
        return {
            "response": (
                "I cannot recommend a drug, dose, or treatment. I can only analyze the "
                "satisfaction expressed in a patient review. Please consult a qualified clinician "
                "for medical decisions."
            ),
            "result": None,
        }
    result = grounded_prediction_semantic(message, drug, condition, k=3)
    label = result["prediction"]["predicted_satisfaction"]
    confidence = max(result["prediction"]["probabilities"].values())
    return {
        "response": f"Predicted patient satisfaction: {label} (model confidence {confidence:.1%}).",
        "result": result,
    }


demo_chat_semantic = review_assistant_semantic(
    "It helped my symptoms, but the nausea was difficult during the first week.",
    drug="example medication",
    condition="example condition",
)
print(demo_chat_semantic["response"])
display(demo_chat_semantic["result"]["prediction"] if demo_chat_semantic["result"] else None)
display(demo_chat_semantic["result"]["similar_reviews"] if demo_chat_semantic["result"] else None)


Predicted patient satisfaction: high (model confidence 48.6%).


{'predicted_satisfaction': np.str_('high'),
 'probabilities': {np.str_('high'): 0.486,
  np.str_('low'): 0.0585,
  np.str_('medium'): 0.4555},
 'evidence_terms': ['helped',
  'nausea',
  'xam',
  'ffi',
  'dur',
  'e_m',
  'usea ',
  'symptoms']}

,source_row,drugName,condition,rating,satisfaction,similarity,review_clean
0,9678,SMZ-TMP DS,Urinary Tract Infection,4,low,0.735604,"""Spent first day sleeping and second too even ..."
1,27188,Vilazodone,Major Depressive Disorde,1,low,0.728926,"""It didn't work for me. At all. I had severe n..."
2,13512,Trulicity,"Diabetes, Type 2",1,low,0.721932,"""Drug works if you can stand the side effects ..."


## 13. Export Artifacts for the Streamlit Chatbot App

The classifier pipeline, the sentence-transformer embeddings for the training set, and the training metadata are saved to `model_artifacts/` so the Streamlit app (`streamlit_app.py`) can load them directly without retraining. Run this cell last, after the pipeline and embeddings above have been fit.

In [ ]:
import joblib

ARTIFACT_DIR = Path("model_artifacts")
ARTIFACT_DIR.mkdir(exist_ok=True)

joblib.dump(prototype, ARTIFACT_DIR / "tfidf_logreg_pipeline.joblib")
np.save(ARTIFACT_DIR / "train_embeddings.npy", train_embeddings)
retrieval_model[["drugName", "condition", "rating", "satisfaction", "review_clean", "model_input"]].to_parquet(
    ARTIFACT_DIR / "train_model.parquet"
)
(ARTIFACT_DIR / "embed_model_name.txt").write_text(EMBED_MODEL_NAME)

print("✓ Improved artifacts exported to:", ARTIFACT_DIR.resolve())
for f in sorted(ARTIFACT_DIR.iterdir()):
    print(" -", f.name)


✓ Improved artifacts exported to: /content/model_artifacts
 - embed_model_name.txt
 - tfidf_logreg_pipeline.joblib
 - train_embeddings.npy
 - train_model.parquet


## 14. Professional Medication Review Intelligence Dashboard

The final interface now behaves like a real review-intelligence product rather than a generic chatbot. Medication, condition, and review are supplied as model context. The classifier is trained on 90,000 reviews, while a separate 35,000-review library keeps semantic retrieval responsive.


In [ ]:
%pip install -q streamlit plotly


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 24.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 77.2 MB/s eta 0:00:00


In [ ]:
%%writefile streamlit_app.py
import html
import re
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import streamlit as st
from sentence_transformers import SentenceTransformer
from sklearn.neighbors import NearestNeighbors

ARTIFACT_DIR = Path(__file__).parent / "model_artifacts"
COLORS = {"low": "#EF6A67", "medium": "#E5A733", "high": "#18A48F"}
SOFT = {"low": "#FFF0EF", "medium": "#FFF8E6", "high": "#EAF9F5"}
LABELS = {"low": "Low satisfaction", "medium": "Medium satisfaction", "high": "High satisfaction"}
ICONS = {"low": "↓", "medium": "—", "high": "↑"}
MEDICAL_ADVICE = re.compile(
    r"\b(should i|can i take|dose|dosage|prescribe|stop taking|best drug|recommend|treatment)\b",
    flags=re.IGNORECASE,
)
EXPERIENCES = {
    "Nausea": r"\b(nausea|nauseous|sick to my stomach)\b",
    "Dizziness": r"\b(dizzy|dizziness|lightheaded)\b",
    "Headache": r"\b(headache|migraine)\b",
    "Fatigue": r"\b(tired|fatigue|exhausted|sleepy)\b",
    "Pain": r"\b(pain|painful|aching|ache)\b",
    "Sleep change": r"\b(insomnia|sleep|slept|awake)\b",
    "Mood change": r"\b(anxiety|anxious|depressed|depression|mood)\b",
    "Improvement": r"\b(helped|effective|improved|better|worked|relief)\b",
    "Worsening": r"\b(worse|worsened|severe|unbearable|terrible)\b",
}

st.set_page_config(page_title="MedReview Insight", page_icon="💊", layout="wide", initial_sidebar_state="expanded")
st.markdown("""
<style>
@import url('https://fonts.googleapis.com/css2?family=DM+Sans:wght@400;500;600;700&display=swap');
:root{--navy:#091F35;--navy2:#123E5B;--teal:#118B83;--mint:#6FDDD0;--ink:#142B3B;--muted:#647B88;--line:#DCE8EB;--surface:#FFF;--canvas:#F4F8F9}
html,body,[class*="css"]{font-family:'DM Sans',sans-serif;color:var(--ink)}
[data-testid="stAppViewContainer"]{background:radial-gradient(circle at 94% 0%,rgba(67,208,192,.13),transparent 25rem),linear-gradient(180deg,#FAFCFD 0%,#F2F7F8 100%)}
[data-testid="stHeader"]{background:rgba(250,252,253,.82);backdrop-filter:blur(10px)}
[data-testid="stSidebar"]{background:linear-gradient(180deg,#F8FCFC,#EAF4F5);border-right:1px solid #D8E6E9}
.block-container{max-width:1450px;padding-top:1.7rem;padding-bottom:4rem}
.hero{position:relative;overflow:hidden;background:linear-gradient(118deg,#071D33 0%,#123E5B 58%,#08777A 100%);color:white;border-radius:28px;padding:2.25rem 2.55rem 2rem;box-shadow:0 24px 58px rgba(11,34,53,.22);margin-bottom:1rem}
.hero:after{content:"";position:absolute;width:330px;height:330px;right:-75px;top:-145px;border-radius:50%;border:55px solid rgba(117,235,220,.10)}
.kicker{color:#8CEADF;font-size:.74rem;font-weight:700;letter-spacing:.17em;text-transform:uppercase}
.hero h1{color:white!important;font-size:2.5rem;letter-spacing:-.035em;margin:.35rem 0 .4rem}
.hero p{color:#D8EEF0;margin:.2rem 0;max-width:850px}
.chips{display:flex;gap:.45rem;flex-wrap:wrap;margin-top:1rem}
.chip{border:1px solid rgba(255,255,255,.18);background:rgba(255,255,255,.09);border-radius:999px;padding:.3rem .68rem;font-size:.74rem;color:#F2FFFF}
.notice{background:#FFF9E8;border:1px solid #F0DA9B;border-left:4px solid #E5A733;border-radius:12px;padding:.7rem 1rem;color:#6F5919;margin:.7rem 0 1.1rem}
.hero-subtitle{font-size:1.16rem!important;font-weight:700;color:#D9FAF7!important;margin:.15rem 0 .28rem!important}
.hero-support{font-size:.96rem!important;color:#D6E6EC!important;margin:0 0 .55rem!important}
.hero-team{font-size:.78rem!important;color:#AFC8D2!important;margin:.2rem 0 0!important;letter-spacing:.01em}
.sidebar-warning{font-size:.76rem;line-height:1.48;color:#667985;background:#F4F8F9;border:1px solid #DDE9EC;border-radius:12px;padding:.72rem .8rem;margin-top:1rem}
.metric-grid{display:grid;grid-template-columns:repeat(4,1fr);gap:.8rem;margin:.2rem 0 1.2rem}
.metric-card{background:white;border:1px solid var(--line);border-radius:17px;padding:1rem 1.1rem;box-shadow:0 8px 25px rgba(21,58,75,.055)}
.metric-label{font-size:.72rem;color:#6D818C;text-transform:uppercase;letter-spacing:.08em;font-weight:700}
.metric-value{font-size:1.48rem;font-weight:700;color:#113A55;margin-top:.2rem}
.section-card{background:white;border:1px solid var(--line);border-radius:20px;padding:1.25rem 1.35rem;box-shadow:0 10px 28px rgba(21,58,75,.06);margin:.7rem 0}
.result-card{border-radius:19px;padding:1.2rem;text-align:center;border:1px solid;min-height:190px;display:flex;flex-direction:column;justify-content:center}
.result-icon{font-size:2rem;font-weight:700}.result-label{font-size:1.34rem;font-weight:700;margin:.25rem 0}.result-confidence{color:#627986;font-size:.9rem}
.quality-bar{height:8px;background:#E7EFF1;border-radius:999px;margin:.75rem auto .2rem;overflow:hidden;max-width:210px}.quality-fill{height:100%;border-radius:999px}
.evidence-pill{display:inline-block;border:1px solid #CDE3E3;background:#EFF9F7;color:#176F69;border-radius:999px;padding:.3rem .62rem;margin:.18rem;font-size:.78rem;font-weight:600}
.experience-pill{display:inline-block;background:#F2F6F8;color:#345468;border:1px solid #DAE6EA;border-radius:9px;padding:.34rem .58rem;margin:.2rem;font-size:.78rem}
.review-card{background:white;border:1px solid var(--line);border-radius:17px;padding:1rem 1.1rem;margin:.7rem 0;box-shadow:0 7px 20px rgba(21,58,75,.045)}
.review-meta{color:#667D89;font-size:.78rem;margin-bottom:.5rem}.source-badge{display:inline-block;background:#123E5B;color:white;border-radius:999px;padding:.18rem .52rem;margin-right:.4rem;font-size:.7rem;font-weight:700}
.grounded-answer{background:linear-gradient(135deg,#EDF9F6,#F4FAFC);border:1px solid #CFE7E4;border-radius:16px;padding:1rem 1.15rem;color:#234557}
.empty-state{text-align:center;padding:3.2rem 1rem;color:#6A7F8A}.empty-icon{font-size:2.6rem;margin-bottom:.6rem}
.sidebar-logo{font-size:1.12rem;font-weight:700;color:#103B56;margin-bottom:.2rem}.sidebar-sub{font-size:.78rem;color:#6C818B;margin-bottom:1rem}
.stButton>button{border-radius:11px!important;border:1px solid #CFE0E4!important;font-weight:600!important}
.stButton>button[kind="primary"]{background:linear-gradient(135deg,#0F7D76,#16A18F)!important;border:0!important;color:white!important;box-shadow:0 7px 17px rgba(15,125,118,.18)}
[data-testid="stTextInput"] input,[data-testid="stTextArea"] textarea{border-radius:12px!important;border-color:#CFE0E4!important;background:rgba(255,255,255,.93)!important}
[data-baseweb="tab-list"]{gap:.35rem;background:#E9F1F3;padding:.3rem;border-radius:14px}
[data-baseweb="tab"]{border-radius:10px;padding:.6rem 1rem}[aria-selected="true"][data-baseweb="tab"]{background:white;box-shadow:0 3px 10px rgba(21,58,75,.08)}
@media(max-width:900px){.metric-grid{grid-template-columns:repeat(2,1fr)}.hero{padding:1.7rem}.hero h1{font-size:2rem}}
</style>
""", unsafe_allow_html=True)

# Presentation-only visual enhancements. CSS animations do not affect model runtime.
st.markdown("""
<style>
.hero{border-radius:26px;padding:1.5rem 2.25rem 1.35rem;box-shadow:0 22px 56px rgba(7,29,51,.24);animation:heroEntrance .65s ease-out}
.hero h1{font-size:2.55rem;margin:.12rem 0 .3rem}
.hero:before{content:"";position:absolute;width:380px;height:380px;right:-110px;top:-180px;border-radius:50%;background:radial-gradient(circle,rgba(111,221,208,.28),rgba(111,221,208,.04) 65%,transparent 70%);animation:floatingGlow 7s ease-in-out infinite}
.hero h1,.hero p,.hero .kicker,.hero .chips{position:relative;z-index:2}.hero-copy{position:relative;z-index:3;max-width:calc(100% - 310px)}
.hero-visual{position:absolute;z-index:2;right:1.8rem;top:50%;width:270px;height:150px;transform:translateY(-50%);filter:drop-shadow(0 16px 22px rgba(0,0,0,.18));animation:visualFloat 5.5s ease-in-out infinite}
.hero-visual .glass{fill:rgba(255,255,255,.09);stroke:rgba(187,255,247,.36);stroke-width:1.5}.hero-visual .shield{fill:url(#shieldGradient);stroke:rgba(255,255,255,.54);stroke-width:1.5}.hero-visual .cross{fill:#E8FFFC}.hero-visual .pulse-line{fill:none;stroke:#8CF5E8;stroke-width:4;stroke-linecap:round;stroke-linejoin:round;stroke-dasharray:240;stroke-dashoffset:240;animation:drawPulse 3.1s ease-in-out infinite}.hero-visual .orb{fill:#8CF5E8;animation:orbGlow 2.4s ease-in-out infinite}.hero-visual .orb:nth-of-type(2){animation-delay:.7s}.hero-visual .orb:nth-of-type(3){animation-delay:1.3s}
@keyframes heroEntrance{from{opacity:0;transform:translateY(-14px)}to{opacity:1;transform:translateY(0)}}
@keyframes floatingGlow{0%,100%{transform:translateY(0) scale(1)}50%{transform:translateY(18px) scale(1.07)}}
.workflow-ribbon{position:relative;display:grid;grid-template-columns:repeat(4,1fr);background:rgba(255,255,255,.9);border:1px solid var(--line);border-radius:18px;margin:.3rem 0 1.2rem;overflow:hidden;box-shadow:0 10px 28px rgba(21,58,75,.07)}
.workflow-ribbon:before{content:"";position:absolute;left:-35%;bottom:0;width:35%;height:3px;background:linear-gradient(90deg,transparent,#45C8D4,#8CF5E8,transparent);animation:workflowSweep 4.2s ease-in-out infinite}
.workflow-step{position:relative;padding:.95rem 1rem;text-align:center}.workflow-step:not(:last-child):after{content:"›";position:absolute;right:-5px;top:50%;transform:translateY(-50%);color:#8AA1AB;font-size:1.5rem;z-index:3}
.workflow-number{display:inline-flex;align-items:center;justify-content:center;width:26px;height:26px;margin-right:.4rem;border-radius:50%;background:linear-gradient(135deg,#0F8B83,#45C8D4);color:white;font-size:.75rem;font-weight:700;animation:stepBreathe 4s ease-in-out infinite}.workflow-step:nth-child(2) .workflow-number{animation-delay:.5s}.workflow-step:nth-child(3) .workflow-number{animation-delay:1s}.workflow-step:nth-child(4) .workflow-number{animation-delay:1.5s}.workflow-label{color:#23485D;font-size:.82rem;font-weight:700}
.metric-card{position:relative;overflow:hidden;background:linear-gradient(145deg,#FFF,#F8FCFC);border-radius:19px;transition:transform .18s ease,box-shadow .18s ease,border-color .18s ease}
.metric-card:after{content:"";position:absolute;width:75px;height:75px;right:-35px;top:-35px;border-radius:50%;background:rgba(69,200,212,.12)}
.metric-card{animation:metricRise .55s both}.metric-card:nth-child(2){animation-delay:.08s}.metric-card:nth-child(3){animation-delay:.16s}.metric-card:nth-child(4){animation-delay:.24s}.metric-card:hover{transform:translateY(-4px);border-color:#9FDAD5;box-shadow:0 15px 34px rgba(21,58,75,.11)}
.section-card{animation:cardEntrance .45s ease-out}@keyframes cardEntrance{from{opacity:0;transform:translateY(8px)}to{opacity:1;transform:translateY(0)}}
.result-card{border-radius:22px;box-shadow:0 12px 30px rgba(21,58,75,.08);animation:resultPulse .55s ease-out}@keyframes resultPulse{from{opacity:0;transform:scale(.96)}to{opacity:1;transform:scale(1)}}
.quality-fill{animation:progressAnimation 1s ease-out}@keyframes progressAnimation{from{width:0}}
.evidence-pill,.review-card,.stButton>button{transition:transform .16s ease,box-shadow .16s ease}.evidence-pill:hover{transform:translateY(-2px);box-shadow:0 5px 13px rgba(23,111,105,.12)}
.review-card{border-left:4px solid #168F88}.review-card:hover{transform:translateX(4px);box-shadow:0 13px 30px rgba(21,58,75,.10)}
.source-badge{background:linear-gradient(135deg,#123E5B,#168F88)}
.empty-state{border:1px dashed #C9DBDF;border-radius:20px;background:linear-gradient(135deg,rgba(255,255,255,.65),rgba(235,247,247,.68))}.empty-icon{color:#168F88;animation:emptyFloat 3s ease-in-out infinite}@keyframes emptyFloat{0%,100%{transform:translateY(0)}50%{transform:translateY(-6px)}}
.stButton>button:hover{transform:translateY(-2px);box-shadow:0 7px 17px rgba(21,58,75,.10)}
.stButton>button[kind="primary"]{position:relative;overflow:hidden}.stButton>button[kind="primary"]:after{content:"";position:absolute;top:0;left:-120%;width:55%;height:100%;background:linear-gradient(90deg,transparent,rgba(255,255,255,.28),transparent);transform:skewX(-20deg);animation:buttonShine 4.8s ease-in-out infinite}
@keyframes visualFloat{0%,100%{transform:translateY(-50%)}50%{transform:translateY(calc(-50% - 8px))}}
@keyframes drawPulse{0%{stroke-dashoffset:240;opacity:.25}35%,72%{stroke-dashoffset:0;opacity:1}100%{stroke-dashoffset:-240;opacity:.2}}
@keyframes orbGlow{0%,100%{opacity:.35;transform:scale(.82)}50%{opacity:1;transform:scale(1.2)}}
@keyframes workflowSweep{0%{left:-35%;opacity:0}15%{opacity:1}75%{opacity:1}100%{left:100%;opacity:0}}
@keyframes stepBreathe{0%,100%{box-shadow:0 0 0 0 rgba(69,200,212,0)}20%{box-shadow:0 0 0 7px rgba(69,200,212,.13)}40%{box-shadow:0 0 0 0 rgba(69,200,212,0)}}
@keyframes metricRise{from{opacity:0;transform:translateY(12px)}to{opacity:1;transform:translateY(0)}}
@keyframes buttonShine{0%,68%{left:-120%}88%,100%{left:150%}}
@media(max-width:900px){.workflow-ribbon{grid-template-columns:repeat(2,1fr)}.hero-copy{max-width:100%}.hero-visual{display:none}}
@media(prefers-reduced-motion:reduce){*,*:before,*:after{animation-duration:.01ms!important;animation-iteration-count:1!important;scroll-behavior:auto!important}}
</style>
""", unsafe_allow_html=True)

def clean_text(value):
    return re.sub(r"\s+", " ", html.unescape(str(value))).strip()

@st.cache_resource(show_spinner="Loading review intelligence models…")
def load_artifacts():
    required = ["tfidf_logreg_pipeline.joblib", "train_embeddings.npy", "train_model.parquet", "embed_model_name.txt"]
    missing = [name for name in required if not (ARTIFACT_DIR / name).exists()]
    if missing:
        raise FileNotFoundError("Missing artifacts: " + ", ".join(missing) + ". Run Section 13 first.")
    pipeline = joblib.load(ARTIFACT_DIR / required[0])
    embeddings = np.load(ARTIFACT_DIR / required[1])
    corpus = pd.read_parquet(ARTIFACT_DIR / required[2])
    embedder = SentenceTransformer((ARTIFACT_DIR / required[3]).read_text().strip())
    retriever = NearestNeighbors(n_neighbors=5, metric="cosine", algorithm="brute").fit(embeddings)
    return pipeline, embedder, retriever, corpus

try:
    pipeline, embedder, retriever, corpus = load_artifacts()
except Exception as exc:
    st.error(str(exc)); st.stop()

def model_text(review, drug, condition):
    return f"drug_{clean_text(drug).replace(' ','_')} condition_{clean_text(condition).replace(' ','_')} {clean_text(review)}"

def evidence_terms(text, label, top_n=8):
    vectorizer, classifier = pipeline.named_steps["tfidf"], pipeline.named_steps["clf"]
    row = vectorizer.transform([text])
    if row.nnz == 0: return []
    names = np.asarray(vectorizer.get_feature_names_out())
    idx = list(classifier.classes_).index(label)
    contribution = row.data * classifier.coef_[idx, row.indices]
    selected = names[row.indices[np.argsort(contribution)[::-1][:top_n]]]
    return [str(term).split("__", 1)[-1] for term in selected]

def extract_experiences(review):
    text = clean_text(review).lower()
    return [name for name, pattern in EXPERIENCES.items() if re.search(pattern, text, re.I)]

def analyze(review, drug, condition, k=3):
    text = model_text(review, drug, condition)
    raw_probs = pipeline.predict_proba([text])[0]
    probabilities = dict(zip(pipeline.classes_, map(float, raw_probs)))
    label = max(probabilities, key=probabilities.get)
    query = embedder.encode([text], normalize_embeddings=True)
    distances, indices = retriever.kneighbors(query, n_neighbors=min(k, len(corpus)))
    similar = corpus.iloc[indices[0]].copy()
    similar["similarity"] = np.clip(1 - distances[0], 0, 1)
    return {
        "review": review, "drug": drug, "condition": condition, "label": label,
        "probabilities": probabilities, "terms": evidence_terms(text, label),
        "experiences": extract_experiences(review), "similar": similar,
        "retrieval_quality": float(similar["similarity"].mean()),
    }

def probability_chart(probabilities):
    order = ["low", "medium", "high"]
    fig = go.Figure(go.Bar(
        x=[probabilities.get(x, 0) for x in order], y=[LABELS[x] for x in order], orientation="h",
        marker=dict(color=[COLORS[x] for x in order], line=dict(width=0)),
        text=[f"{probabilities.get(x, 0):.1%}" for x in order], textposition="outside",
    ))
    fig.update_layout(height=270, margin=dict(l=10, r=40, t=18, b=20),
        xaxis=dict(tickformat=".0%", range=[0, 1.08], title="Model probability", gridcolor="#E8EFF1"),
        yaxis=dict(autorange="reversed"), plot_bgcolor="rgba(0,0,0,0)", paper_bgcolor="rgba(0,0,0,0)", showlegend=False)
    return fig

def distribution_chart(frame, field, title):
    counts = frame[field].value_counts().reindex(["low", "medium", "high"], fill_value=0)
    fig = go.Figure(go.Pie(labels=[LABELS[x] for x in counts.index], values=counts.values, hole=.64,
        marker=dict(colors=[COLORS[x] for x in counts.index], line=dict(color="white", width=3)), textinfo="percent"))
    fig.update_layout(title=dict(text=title, font=dict(size=14)), height=300, margin=dict(l=10, r=10, t=45, b=10),
        paper_bgcolor="rgba(0,0,0,0)", legend=dict(orientation="h", y=-.08))
    return fig

def metric_cards(items):
    cards = "".join(f'<div class="metric-card"><div class="metric-label">{html.escape(label)}</div><div class="metric-value">{html.escape(str(value))}</div></div>' for label, value in items)
    st.markdown(f'<div class="metric-grid">{cards}</div>', unsafe_allow_html=True)

def render_review_cards(similar):
    for i, (_, row) in enumerate(similar.iterrows(), 1):
        excerpt = clean_text(row["review_clean"])
        if len(excerpt) > 430: excerpt = excerpt[:430].rstrip() + "…"
        st.markdown(
            f'<div class="review-card"><div class="review-meta"><span class="source-badge">SOURCE {i}</span>'
            f'<b>{html.escape(str(row["drugName"]))}</b> · {html.escape(str(row["condition"]))} · '
            f'Rating {row["rating"]}/10 · Semantic match {row["similarity"]:.0%}</div>'
            f'<div>{html.escape(excerpt)}</div></div>', unsafe_allow_html=True)

def grounded_summary(result):
    label, confidence = result["label"], result["probabilities"][result["label"]]
    evidence = ", ".join(result["terms"][:4]) if result["terms"] else "the overall language pattern"
    experiences = ", ".join(result["experiences"][:4]) if result["experiences"] else "no common experience category was explicitly detected"
    return (f"This review is classified as **{LABELS[label].lower()}** with **{confidence:.1%} confidence**. "
            f"The strongest model signals include **{evidence}**. Detected experience themes: **{experiences}**. "
            f"The explanation is grounded in {len(result['similar'])} semantically similar patient reviews.")

def answer_followup(question, result):
    q = question.lower(); label = result["label"]; confidence = result["probabilities"][label]
    if "why" in q or "classif" in q:
        terms = ", ".join(result["terms"][:6]) or "the review's overall wording"
        return f"The model selected **{LABELS[label].lower()}** at **{confidence:.1%} confidence**. Its strongest contributing language signals were: {terms}."
    if "side effect" in q or "symptom" in q or "experience" in q:
        found = ", ".join(result["experiences"]) or "No common symptom or experience category was explicitly detected."
        return f"Detected experience themes: **{found}**. These are text mentions, not verified clinical findings."
    if "similar" in q or "source" in q or "evidence" in q:
        best = result["similar"].iloc[0]
        return f"The strongest retrieved source is a review for **{best['drugName']}** and **{best['condition']}**, with {best['similarity']:.0%} semantic similarity and a {best['rating']}/10 rating. See all source cards above."
    if "confiden" in q or "uncertain" in q:
        spread = sorted(result["probabilities"].values(), reverse=True)
        margin = spread[0] - spread[1]
        return f"Confidence is **{confidence:.1%}** and the margin over the second class is **{margin:.1%}**. A smaller margin means the language is more mixed or ambiguous."
    return grounded_summary(result)

st.markdown("""<div class="hero"><div class="hero-copy"><div class="kicker">PATIENT-REVIEW INTELLIGENCE</div><h1>💊 MedReview Insight</h1><p class="hero-subtitle">AI-powered analysis of medication experiences</p><p class="hero-support">Predict satisfaction, uncover experience themes, and retrieve similar patient reviews.</p><p class="hero-team">ISBA 2411 · Group 1 · Zahra Fahimfar, Varsha Pai, Krystle Jozen Dario</p></div><svg class="hero-visual" viewBox="0 0 300 170" role="img" aria-label="Animated healthcare intelligence illustration"><defs><linearGradient id="shieldGradient" x1="0" y1="0" x2="1" y2="1"><stop offset="0" stop-color="#65E0D2"/><stop offset="1" stop-color="#168F9A"/></linearGradient></defs><rect class="glass" x="8" y="18" width="284" height="134" rx="28"/><circle class="orb" cx="35" cy="44" r="5"/><circle class="orb" cx="267" cy="47" r="4"/><circle class="orb" cx="251" cy="132" r="5"/><path class="shield" d="M150 36 L193 53 V83 C193 111 175 130 150 140 C125 130 107 111 107 83 V53 Z"/><rect class="cross" x="141" y="63" width="18" height="52" rx="5"/><rect class="cross" x="124" y="80" width="52" height="18" rx="5"/><path class="pulse-line" d="M28 101 H67 L78 79 L92 121 L104 101 H121 M179 101 H197 L207 84 L219 114 L230 101 H271"/></svg></div>""", unsafe_allow_html=True)
st.markdown("""
<div class="workflow-ribbon">
  <div class="workflow-step"><span class="workflow-number">1</span><span class="workflow-label">Analyze</span></div>
  <div class="workflow-step"><span class="workflow-number">2</span><span class="workflow-label">Explain</span></div>
  <div class="workflow-step"><span class="workflow-number">3</span><span class="workflow-label">Retrieve</span></div>
  <div class="workflow-step"><span class="workflow-number">4</span><span class="workflow-label">Ground</span></div>
</div>
""", unsafe_allow_html=True)

def clear_workspace():
    # Widget keys must be reset explicitly so Streamlit does not restore their
    # previous browser values on the following rerun.
    st.session_state["drug"] = ""
    st.session_state["condition"] = ""
    st.session_state["review_input"] = ""
    st.session_state.pop("analysis", None)
    st.session_state.pop("followups", None)

with st.sidebar:
    st.markdown('<div class="sidebar-logo">MedReview Workspace</div><div class="sidebar-sub">Add the medication and condition context used for analysis and retrieval.</div>', unsafe_allow_html=True)
    drug = st.text_input("Drug name", key="drug", placeholder="e.g., Lisinopril")
    condition = st.text_input("Condition", key="condition", placeholder="e.g., High Blood Pressure")
    k = st.slider("Evidence sources", 2, 5, 3)
    st.markdown("---")
    st.caption("Prediction: hybrid word/character TF-IDF. Retrieval: MiniLM semantic embeddings.")
    st.button("Clear workspace", use_container_width=True, on_click=clear_workspace)
    st.markdown('<div class="sidebar-warning">⚕️ <b>Research and educational use only.</b><br>Not medical advice. Results summarize patient-reported experiences and model predictions.</div>', unsafe_allow_html=True)

metric_cards([
    ("Model training reviews", "90,000"),
    ("Semantic search library", f"{len(corpus):,}"),
    ("Medications represented", f"{corpus['drugName'].nunique():,}"),
    ("Average rating", f"{corpus['rating'].mean():.1f} / 10"),
])

tab_analysis, tab_explore, tab_method = st.tabs(["Review Analysis", "Explore Review Data", "How It Works"])

with tab_analysis:
    def use_example(sample_drug, sample_condition, sample_text):
        st.session_state.drug = sample_drug
        st.session_state.condition = sample_condition
        st.session_state.review_input = sample_text

    left, right = st.columns([1.45, .75])
    with left:
        review = st.text_area(
            "Patient medication review",
            height=155,
            key="review_input",
            placeholder="Describe the medication experience, including benefits, side effects, or concerns…",
        )
    with right:
        st.markdown("**Try an example**")
        examples = [
            ("Side effects", "Lisinopril", "High Blood Pressure", "It reduced my symptoms, but the nausea and dizziness were difficult during the first week."),
            ("Strong benefit", "Sertraline", "Depression", "This worked extremely well and I finally felt like myself again with no major side effects."),
            ("Mixed outcome", "Gabapentin", "Neuropathic Pain", "I noticed a small improvement, although my sleep became worse and I still had some pain."),
        ]
        for i, (example_label, sample_drug, sample_condition, sample_text) in enumerate(examples):
            st.button(
                example_label,
                key=f"example_{i}",
                use_container_width=True,
                on_click=use_example,
                args=(sample_drug, sample_condition, sample_text),
            )
    if st.button("Analyze review", type="primary", use_container_width=True):
        if not clean_text(review): st.warning("Enter a review before running the analysis.")
        elif MEDICAL_ADVICE.search(review): st.warning("Please describe an experience rather than requesting medication or dosage advice.")
        else:
            with st.spinner("Analyzing satisfaction signals and retrieving grounded evidence…"):
                st.session_state.analysis = analyze(review, drug or "unknown drug", condition or "unknown condition", k)

    result = st.session_state.get("analysis")
    if not result:
        st.markdown('<div class="empty-state"><div class="empty-icon">⌁</div><b>Your review intelligence report will appear here.</b><br>Enter a patient review and select Analyze review.</div>', unsafe_allow_html=True)
    else:
        label, confidence = result["label"], result["probabilities"][result["label"]]
        metric_cards([
            ("Prediction", LABELS[label]), ("Confidence", f"{confidence:.1%}"),
            ("Evidence quality", f"{result['retrieval_quality']:.0%}"), ("Grounded sources", len(result["similar"])),
        ])
        c1, c2 = st.columns([.75, 1.5])
        with c1:
            st.markdown(f'<div class="result-card" style="background:{SOFT[label]};border-color:{COLORS[label]}55"><div class="result-icon" style="color:{COLORS[label]}">{ICONS[label]}</div><div class="result-label" style="color:{COLORS[label]}">{LABELS[label]}</div><div class="result-confidence">Model confidence {confidence:.1%}</div><div class="quality-bar"><div class="quality-fill" style="width:{confidence*100:.0f}%;background:{COLORS[label]}"></div></div></div>', unsafe_allow_html=True)
        with c2: st.plotly_chart(probability_chart(result["probabilities"]), use_container_width=True)
        st.markdown('<div class="section-card"><b>Grounded interpretation</b><br><br>' + grounded_summary(result) + '</div>', unsafe_allow_html=True)
        ev1, ev2 = st.columns(2)
        with ev1:
            st.markdown("#### Language evidence")
            pills = "".join(f'<span class="evidence-pill">{html.escape(term)}</span>' for term in result["terms"])
            st.markdown(pills or "No high-weight evidence term identified.", unsafe_allow_html=True)
        with ev2:
            st.markdown("#### Experience themes")
            pills = "".join(f'<span class="experience-pill">{html.escape(term)}</span>' for term in result["experiences"])
            st.markdown(pills or "No common experience category explicitly detected.", unsafe_allow_html=True)
        st.markdown("#### Similar patient-review evidence")
        st.caption("Retrieved sources are patient anecdotes and are not clinical evidence.")
        render_review_cards(result["similar"])
        st.markdown("#### Ask about this result")
        questions = ["Why this classification?", "Which experiences were mentioned?", "What is the strongest similar review?", "How certain is the model?"]
        qcols = st.columns(4)
        for i, question in enumerate(questions):
            if qcols[i].button(question, key=f"follow_{i}", use_container_width=True):
                st.session_state.followups = [(question, answer_followup(question, result))]
        for question, answer in st.session_state.get("followups", []):
            st.markdown(f'<div class="grounded-answer"><b>{html.escape(question)}</b><br>{answer}</div>', unsafe_allow_html=True)

with tab_explore:
    filtered = corpus.copy()
    if drug and drug.lower() != "unknown drug": filtered = filtered[filtered["drugName"].str.contains(re.escape(drug), case=False, na=False)]
    if condition and condition.lower() != "unknown condition": filtered = filtered[filtered["condition"].str.contains(re.escape(condition), case=False, na=False)]
    st.caption("This tab summarizes the 35,000-review semantic search library. The classifier itself was trained on 90,000 reviews.")
    if filtered.empty:
        st.info("No matching review records were found. Try a broader drug or condition name in the sidebar.")
    else:
        metric_cards([("Searchable reviews", f"{len(filtered):,}"), ("Average rating", f"{filtered['rating'].mean():.1f}"), ("Medications", filtered["drugName"].nunique()), ("Conditions", filtered["condition"].nunique())])
        p1, p2 = st.columns(2)
        with p1: st.plotly_chart(distribution_chart(filtered, "satisfaction", "Satisfaction distribution"), use_container_width=True)
        with p2:
            rating_counts = filtered["rating"].value_counts().sort_index()
            fig = go.Figure(go.Bar(x=rating_counts.index, y=rating_counts.values, marker_color="#168F88"))
            fig.update_layout(title="Rating distribution", height=300, margin=dict(l=20,r=15,t=45,b=25), xaxis_title="Patient rating", yaxis_title="Reviews", plot_bgcolor="rgba(0,0,0,0)", paper_bgcolor="rgba(0,0,0,0)")
            st.plotly_chart(fig, use_container_width=True)
        st.markdown("#### Most represented medications")
        top = filtered["drugName"].value_counts().head(10).rename_axis("Medication").reset_index(name="Reviews")
        st.dataframe(top, use_container_width=True, hide_index=True)

with tab_method:
    st.markdown("""
    ### A grounded research workflow

    1. **Contextualize:** combine medication name, condition, and patient review.
    2. **Predict:** classify satisfaction with word- and character-level TF-IDF plus a balanced averaged SGD logistic classifier.
    3. **Explain:** identify the highest-contributing text features for the selected class.
    4. **Retrieve:** use MiniLM embeddings to search 35,000 semantically similar patient reviews.
    5. **Ground:** show source cards and provide deterministic follow-up explanations based only on model outputs and retrieved reviews.

    **Technology:** hybrid word- and character-level TF-IDF, balanced averaged SGD logistic classification, and MiniLM semantic retrieval.

    This design avoids a large generative model, so the interface remains lightweight and does not add material training time. Patient reviews may be noisy or inconsistent with their numeric ratings; results should not be used for medical decisions.
    """)


Writing streamlit_app.py


### 14.1 Launch the app

Run the next cell after artifact export. It starts Streamlit and displays the professional app directly inside Colab. No ngrok token is required.


In [ ]:
import subprocess, sys, time
import requests
from IPython.display import HTML, display
from google.colab.output import eval_js

try:
    streamlit_proc.terminate()
except Exception:
    pass

log_path = "/content/streamlit_logs.txt"
streamlit_proc = subprocess.Popen(
    [sys.executable, "-m", "streamlit", "run", "streamlit_app.py",
     "--server.port", "8501", "--server.address", "0.0.0.0",
     "--server.headless", "true", "--server.enableCORS", "false",
     "--server.enableXsrfProtection", "false", "--browser.gatherUsageStats", "false"],
    stdout=open(log_path, "w"), stderr=subprocess.STDOUT,
)
healthy = False
for _ in range(60):
    if streamlit_proc.poll() is not None:
        break
    try:
        response = requests.get("http://127.0.0.1:8501/_stcore/health", timeout=1)
        if response.ok:
            healthy = True
            break
    except requests.RequestException:
        pass
    time.sleep(1)

if not healthy:
    print(Path(log_path).read_text()[-6000:])
    raise RuntimeError("Streamlit did not become healthy. The actual error is printed above.")

app_url = eval_js("google.colab.kernel.proxyPort(8501)")
print("✓ Streamlit is healthy. Click the button below (do not use an Untitled blank tab):")
display(HTML(
    f'<a href="{app_url}" target="_blank" '
    'style="display:inline-block;background:#0E7C75;color:white;padding:14px 22px;'
    'border-radius:10px;text-decoration:none;font-weight:700;font-size:16px">'
    'Open MedReview Insight ↗</a>'
))
print("Direct link:", app_url)


✓ Streamlit is healthy. Click the button below (do not use an Untitled blank tab):


Direct link: https://8501-gpu-t4-s-kkb-usw1b2-2dby0a9bewvg9-b.us-west1-2.prod.colab.dev


In [ ]:
# Optional: inspect recent Streamlit logs if the app does not appear.
print(Path("/content/streamlit_logs.txt").read_text()[-4000:])


2026-08-17 01:15:40.314 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.11.216.59:8501




In [ ]:
# Optional: stop the app.
# streamlit_proc.terminate()


In [ ]:
# Security note: no ngrok token is stored in this notebook.
# The Colab port proxy above is the supported launch path for this project.


In [ ]:
print("Notebook complete. Use Section 14.1 to open the app.")


Notebook complete. Use Section 14.1 to open the app.


In [ ]:
!curl -s http://localhost:8501/_stcore/health

ok

**Run notes**

- Use a T4 GPU for faster MiniLM encoding.
- Upload the two Drugs.com CSV files; names with or without `(1)` are accepted.
- Run cells in order. The app launches only after the improved model and retrieval artifacts are exported.
- Predictive performance is reported on the untouched official test split sample using both accuracy and macro-F1.
